# Create Mott Foundation Grants (GRANT PATTERN, method-5 static HTML)

Charles Stewart Mott Foundation grants from the foundation's own WordPress-published grants archive at mott.org. Major US foundation funding civil society, environment, education, and Flint-area community development.

**Prerequisites:**
- Run `scripts/local/mott_to_s3.py` first.

**Data source:** 3 WordPress sitemap files enumerate **3,000 grant URLs**:
- `/grant-sitemap.xml` (1,000 URLs)
- `/grant-sitemap2.xml` (1,000 URLs)
- `/grant-sitemap3.xml` (1,000 URLs)

Per-grant detail page (~50KB) has a `<ul class="list1-entries">` block with `<li><strong>Label</strong><span>Value</span></li>` pairs exposing every field cleanly: Program, Program Initiative, Grant Amount, Grant Period, Location, Geographic Focus.

**S3 location:** `s3a://openalex-ingest/awards/mott/mott_grants.parquet`

**Awarding body:**
- funder_id: 4320307861
- display_name: "Charles Stewart Mott Foundation"
- country: US
- ROR: none
- DOI: 10.13039/100004428

**Coverage (smoke test, 10 grants, 2026-05-22):**
- 100% on title / slug / recipient / program / amount / start_year / location
- 10/10 unique slug → funder_award_id (`mott-{slug}`, slug = `{year}-{numeric-id}`)
- Amount range $36,550 - $17,541,792 (median $200k)

Full corpus crawl (~15 min at 0.3s throttle for 3,000 pages) was running locally at PR-open time; admin re-running on their side will validate end-to-end.

**Programs (visible in `program` field):**
- Environment
- Civil Society
- Education
- Flint Area
- Charles Stewart Mott Foundation (general)

**Provenance:** `mott_grants` (direct from foundation, not an aggregator).

**Priority:** 113 (UCOP 106 / Rita Allen 107 / Schmidt Sciences 108 / NOMIS 109 / Wenner-Gren 110 / KAW 111 / Helmsley 112 are immediately-prior slots in flight; Vilcek at 105 is current main registry head).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mott_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/mott/mott_grants.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.mott_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.mott_raw;


## Step 1.5: Inspect Raw Data, Money Scan, Native Key

Mott publishes per-grant USD amounts on every detail page. Currency is hardcoded USD.

In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.mott_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.mott_raw)
WHERE LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT title, recipient, program, initiative, amount, currency,
       start_date, end_date, location, geographic_focus
FROM openalex.awards.mott_raw LIMIT 5;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS rows_with_amount,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    ROUND(AVG(TRY_CAST(amount AS DOUBLE)), 0) AS avg_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)) / 1e9, 2) AS total_usd_billions
FROM openalex.awards.mott_raw;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.mott_raw;


In [ ]:
%sql
SELECT start_year, COUNT(*) as grants, ROUND(SUM(TRY_CAST(amount AS DOUBLE))/1e6, 1) AS total_usd_m
FROM openalex.awards.mott_raw
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 30;


In [ ]:
%sql
SELECT program, COUNT(*) as cnt, ROUND(SUM(TRY_CAST(amount AS DOUBLE))/1e6, 1) AS total_usd_m
FROM openalex.awards.mott_raw
GROUP BY program ORDER BY total_usd_m DESC LIMIT 20;


In [ ]:
%sql
SELECT recipient, COUNT(*) as cnt, ROUND(SUM(TRY_CAST(amount AS DOUBLE))/1e6, 1) AS total_usd_m
FROM openalex.awards.mott_raw
GROUP BY recipient ORDER BY total_usd_m DESC LIMIT 20;


In [ ]:
%sql
SELECT geographic_focus, COUNT(*) as cnt
FROM openalex.awards.mott_raw
WHERE geographic_focus IS NOT NULL
GROUP BY geographic_focus ORDER BY cnt DESC LIMIT 20;


## Step 1.6: Fail-fast — Verify the Mott Funder Row Exists

**Runbook §2.2.4 mandatory.**

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320307861;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mott_awards
USING delta
AS
WITH
mott_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320307861
),

raw_prepared AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_CAST(start_year AS INT) AS parsed_year,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date
    FROM openalex.awards.mott_raw
    WHERE title IS NOT NULL
      AND TRIM(title) != ''
      AND funder_award_id IS NOT NULL
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.title as display_name,
        g.description as description,

        f.funder_id,
        g.funder_award_id,

        g.parsed_amount as amount,
        'USD' as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'grant' as funding_type,

        -- funder_scheme = "Program / Initiative" when both present, else Program
        CASE
            WHEN g.initiative IS NOT NULL AND TRIM(g.initiative) != '' AND g.program IS NOT NULL AND TRIM(g.program) != ''
                THEN CONCAT(TRIM(g.program), ' / ', TRIM(g.initiative))
            WHEN g.program IS NOT NULL AND TRIM(g.program) != ''
                THEN TRIM(g.program)
            ELSE 'Mott Foundation Grant'
        END as funder_scheme,

        'mott_grants' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        g.parsed_year as start_year,
        YEAR(g.parsed_end_date) as end_year,

        -- lead_investigator: org-level (Mott funds organizations, not individuals).
        -- Per HHMI/Hewlett/Helmsley precedent, recipient name in affiliation slot,
        -- given/family NULL.
        struct(
            CAST(NULL AS STRING) as given_name,
            CAST(NULL AS STRING) as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.recipient), '') as name,
                'US' as country,  -- Mott funds primarily US recipients; some international (Civil Society program)
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN mott_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 113

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mott_grants' AND priority = 113;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    113 as priority  -- Mott priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.mott_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.mott_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.mott_awards;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) as pct_description,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date,
    ROUND(COUNT(end_date) * 100.0 / COUNT(*), 1) as pct_end_date
FROM openalex.awards.mott_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, end_year, funder_scheme,
       amount, currency,
       lead_investigator.affiliation.name AS recipient_org,
       landing_page_url
FROM openalex.awards.mott_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.mott_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as grants, ROUND(SUM(amount)/1e6,1) as total_usd_m
FROM openalex.awards.mott_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;


In [ ]:
%sql
-- §6.7 Amount/currency check. Expected: ~100% pct_amount, USD only.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_amount_nonzero,
    ROUND(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_nonzero_amt,
    ROUND(SUM(amount) / 1e9, 2) AS total_usd_billions
FROM openalex.awards.mott_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount)/1e6,1) as total_usd_m
FROM openalex.awards.mott_awards
GROUP BY funder_scheme ORDER BY total_usd_m DESC LIMIT 20;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.mott_awards;
